# Bernstein–Vazirani in native FlagQuantum

Recover the original five-bit secret 13 from a linear phase oracle. This Notebook uses native Python quantum functions, local synthesis and real executed probabilities. The equivalent phase oracle specializes the original minus-state auxiliary construction. No general quantum callable port or hardware execution is claimed.

The Circuit comparison checks all complex amplitudes, including absolute phase.

See [the native program](main.py), [workflow details](WORKFLOW.md) and [third-party license](../../../CLASSIQ_LICENSE.txt).

## Define the native linear phase oracle

In [ ]:
import math
from collections import Counter
from random import Random
import flagquantum as fq
from flagquantum.drawer import draw
from flagquantum.synth import (
    PHASE,
    Output,
    Program,
    QArray,
    QBit,
    allocate,
    hadamard_transform,
    qfunc,
    synthesize,
)

@qfunc
def linear_phase(weight: float, bit: QBit) -> None:
    PHASE(math.pi * weight, bit)

@qfunc
def hidden_string(weights: list[float], x: QArray[QBit]) -> None:
    for index in range(len(weights)):
        linear_phase(weights[index], x[index])

## Compose the native quantum program

In [ ]:
@qfunc
def main(weights: list[float], x: Output[QArray[QBit]]) -> None:
    allocate(5, x)
    hadamard_transform(x)
    hidden_string(weights, x)
    hadamard_transform(x)

## Bind the original five-bit secret and synthesize

In [ ]:
secret = 13
shots = 1000
weights = [float((secret >> index) & 1) for index in range(5)]
program = synthesize(main, classical_inputs={"weights": weights}, max_qubits=6)

## Draw the synthesized circuit

In [ ]:
print(draw(program.circuit))

## Execute and draw 1,000 local samples

In [ ]:
result = fq.run(program)
probabilities = dict(program.probabilities(result, registers=("x",)))
samples = Random(17).choices(list(probabilities), weights=list(probabilities.values()), k=shots)
counts = Counter(samples)
print("Counts:", dict(counts))

## Recover the secret and verify the complete state

In [ ]:
recovered = counts.most_common(1)[0][0][0]
assert recovered == secret
assert counts == {(secret,): shots}
print("Secret integer:", recovered)
print("Exact recovery probability:", probabilities[(recovered,)])

import torch
from examples.synthesis.algorithms.foundational.bernstein_vazirani.comparisons.circuit import circuit_reference
torch.testing.assert_close(result.statevector(), fq.run(circuit_reference(secret)).statevector(), atol=1e-12, rtol=0)
print("Independent full-state comparison: passed")